In [5]:
import pandas as pd
import os

def sort_large_csv(input_file, output_file, chunk_size=100000):
    """
    Sort a large CSV file based on the 'title' column using chunking for memory efficiency.
    
    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to save sorted CSV file
    chunk_size (int): Number of rows to process at once
    """
    try:
        # Gets total number of rows
        total_rows = sum(1 for _ in open(input_file, 'rb'))
        chunk_count = (total_rows - 1) // chunk_size + 1
        
        print(f"Processing {total_rows:,} rows in {chunk_count} chunks...")
        
        # Creates temporary directory for chunks
        temp_dir = "temp_chunks"
        os.makedirs(temp_dir, exist_ok=True)
        
        # Process file in chunks
        chunk_files = []
        for i, chunk in enumerate(pd.read_csv(input_file, chunksize=chunk_size, 
                                            encoding_errors='replace')):
            print(f"Processing chunk {i+1}/{chunk_count}...")
            
            # Sort current chunk
            chunk_sorted = chunk.sort_values('title')
            
            # Save sorted chunk
            chunk_file = os.path.join(temp_dir, f'chunk_{i}.csv')
            chunk_sorted.to_csv(chunk_file, index=False)
            chunk_files.append(chunk_file)
            
        print("Merging sorted chunks")
        
        # Open all sorted chunks
        chunks = [pd.read_csv(f) for f in chunk_files]
        
        # Merge all chunks and sort
        final_df = pd.concat(chunks, ignore_index=True)
        final_df = final_df.sort_values('title')
        
        # Save final sorted file
        print("Saving final sorted file...")
        final_df.to_csv(output_file, index=False)
        
        # Cleanup temporary files
        print("Cleaning up temporary files")
        for f in chunk_files:
            os.remove(f)
        os.rmdir(temp_dir)
        
        print(f"Successfully sorted and saved to {output_file}")
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        # Cleanup on error
        if os.path.exists(temp_dir):
            for f in os.listdir(temp_dir):
                os.remove(os.path.join(temp_dir, f))
            os.rmdir(temp_dir)

if __name__ == "__main__":
    # Example usage
    input_file = "wikihowAll-1-colm3removed.csv"
    output_file = "wikihowAll-2-sorted&cleaned_from_unhuman_data.csv"
    sort_large_csv(input_file, output_file)

Processing 2,213,841 rows in 23 chunks...


C:\Users\Hp\AppData\Local\Temp\ipykernel_21764\2836250276.py:26: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(input_file, chunksize=chunk_size,


Processing chunk 1/23...


C:\Users\Hp\AppData\Local\Temp\ipykernel_21764\2836250276.py:26: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(input_file, chunksize=chunk_size,


Processing chunk 2/23...
Processing chunk 3/23...
Merging sorted chunks...


C:\Users\Hp\AppData\Local\Temp\ipykernel_21764\2836250276.py:41: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  chunks = [pd.read_csv(f) for f in chunk_files]
C:\Users\Hp\AppData\Local\Temp\ipykernel_21764\2836250276.py:41: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50) have mixed types. Specify dtype option on import or set low_memory=False.
  chunks = [pd.read_csv(f) for f in chunk_files]


Saving final sorted file...
Cleaning up temporary files...
Successfully sorted and saved to wikihowAll-2-sorted.csv


In [6]:
import pandas as pd
import re

def filter_csv_by_ending(input_file, output_file):
    """
    Filters rows to keep only those ending with numbers 1-8
    
    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to save filtered CSV file
    
    Returns:
    tuple: (number of original rows, number of rows after filtering)
    """
    
    # Read CSV in chunks due to large file size
    chunk_size = 10000
    chunks = []
    
    # Pattern to match text ending in numbers 1-8
    pattern = r'.*[1-8]$'
    
    # Process file in chunks
    for chunk in pd.read_csv(input_file, chunksize=chunk_size):
        # Filter rows where either headline or title ends with numbers 1-8
        filtered_chunk = chunk[
            chunk['headline'].str.match(pattern, na=False) |
            chunk['title'].str.match(pattern, na=False)
        ]
        chunks.append(filtered_chunk)
    
    # Combine all filtered chunks
    filtered_df = pd.concat(chunks, ignore_index=True)
    
    # Get row counts
    original_count = sum(1 for _ in pd.read_csv(input_file, chunksize=chunk_size))
    filtered_count = len(filtered_df)
    
    # Save filtered data
    filtered_df.to_csv(output_file, index=False)
    
    return original_count, filtered_count

# Example usage
if __name__ == "__main__":
    input_file = "wikihowAll-2-sorted&cleaned_from_unhuman_data.csv"
    output_file = "wikihowAll-3-removed_single_step_rows.csv"
    
    original, filtered = filter_csv_by_ending(input_file, output_file)
    print(f"Original rows: {original:,}")
    print(f"Filtered rows: {filtered:,}")
    print(f"Removed rows: {original - filtered:,}")

Original rows: 22
Filtered rows: 128,172
Removed rows: -128,150


In [7]:
import pandas as pd
import re

def filter_wikihow_steps(input_file, output_file):
    """
    Filters WikiHow dataset to keep only tasks with more than 2 steps.
    
    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to save filtered CSV file
    """
    
    # Read the CSV file
    df = pd.read_csv(input_file)
    
    # Extract base title (removing the step number) and step number
    df['base_title'] = df['title'].str.replace(r'\d+$', '', regex=True)
    df['step_number'] = df['title'].str.extract(r'(\d+)$').astype(int)
    
    # Count total steps for each task
    step_counts = df.groupby('base_title')['step_number'].max()
    
    # Get base titles with more than 2 steps
    valid_titles = step_counts[step_counts > 2].index
    
    # Filter the dataframe to keep only rows from tasks with more than 2 steps
    filtered_df = df[df['base_title'].isin(valid_titles)]
    
    # Remove the temporary columns we created
    filtered_df = filtered_df.drop(['base_title', 'step_number'], axis=1)
    
    # Save the filtered dataset
    filtered_df.to_csv(output_file, index=False)
    
    return len(df), len(filtered_df)

# Example usage
if __name__ == "__main__":
    input_file = "wikihowAll-3-removed_single_step_rows.csv"
    output_file = "wikihowAll-4-only_rows_with_3or_more_steps.csv"
    
    original_count, filtered_count = filter_wikihow_steps(input_file, output_file)
    
    print(f"Original rows: {original_count:,}")
    print(f"Filtered rows: {filtered_count:,}")
    print(f"Removed rows: {original_count - filtered_count:,}")

Original rows: 128,172
Filtered rows: 106,228
Removed rows: 21,944


In [8]:
import pandas as pd
import re

def format_wikihow_steps(input_file, output_file):
    """
    Restructures WikiHow data to create columns for each step.
    
    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to save restructured CSV file
    """
    try:
        # Read the CSV file
        print("Reading CSV file...")
        df = pd.read_csv(input_file)
        
        # Extract task name and step number from title
        def extract_info(title):
            # Find the last number in the title
            match = re.search(r'(.*?)(\d+)$', title)
            if match:
                task_name = match.group(1).strip()
                step_num = int(match.group(2))
                return pd.Series([task_name, step_num])
            return pd.Series([title, None])
        
        # Add task_name and step_number columns
        df[['task_name', 'step_number']] = df['title'].apply(extract_info)
        
        # Create a dictionary to store the results
        tasks_dict = {}
        
        # Group by task_name and create step columns
        print("Restructuring data...")
        for task_name, group in df.groupby('task_name'):
            steps = {}
            steps['task'] = task_name
            
            # Fill in steps (up to 9)
            for _, row in group.iterrows():
                step_num = row['step_number']
                if step_num and 1 <= step_num <= 9:
                    steps[f'step{step_num}'] = row['headline']
            
            tasks_dict[task_name] = steps
        
        # Convert dictionary to DataFrame
        print("Creating final DataFrame...")
        result_df = pd.DataFrame.from_dict(tasks_dict, orient='index')
        
        # Ensure all step columns exist
        for i in range(1, 10):
            if f'step{i}' not in result_df.columns:
                result_df[f'step{i}'] = None
        
        # Reorder columns
        column_order = ['task'] + [f'step{i}' for i in range(1, 10)]
        result_df = result_df[column_order]
        
        # Reset index and save
        print("Saving restructured data...")
        result_df.reset_index(drop=True).to_csv(output_file, index=False)
        
        print(f"Successfully saved restructured data to {output_file}")
        print(f"Total tasks processed: {len(result_df)}")
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == "__main__":
    input_file = "wikihowAll-4-only_rows_with_3or_more_steps.csv"
    output_file = "wikihowAll-5-dataset before jumbling.csv" # wikihowAll-5-dataset_ground_truth
    format_wikihow_steps(input_file, output_file)

Reading CSV file...
Restructuring data...
Creating final DataFrame...
Saving restructured data...
Successfully saved restructured data to wikihowAll-5-dataset before jumbling.csv
Total tasks processed: 31019


In [9]:
import pandas as pd
import numpy as np

def shuffle_steps(input_file, output_file):
    """
    Randomly shuffles the steps for each task while maintaining task names.
    
    Parameters:
    input_file (str): Path to input CSV file
    output_file (str): Path to save shuffled CSV file
    """
    try:
        # Read the CSV file
        print("Reading CSV file...")
        df = pd.read_csv(input_file)
        
        # Get column names
        step_columns = [col for col in df.columns if col.startswith('step')]
        
        # Create a copy of the DataFrame
        shuffled_df = df.copy()
        
        print("Shuffling steps for each task...")
        # For each row, shuffle the step values
        for idx in range(len(df)):
            # Get current steps (excluding None/NaN values)
            current_steps = df.iloc[idx][step_columns].dropna().tolist()
            
            if current_steps:  # If there are steps to shuffle
                # Shuffle the steps
                shuffled_steps = current_steps.copy()
                np.random.shuffle(shuffled_steps)
                
                # Create a dictionary to update the row
                update_dict = {}
                for i, step in enumerate(shuffled_steps):
                    update_dict[f'step{i+1}'] = step
                
                # Fill remaining steps with None if original had fewer than 9 steps
                for i in range(len(shuffled_steps) + 1, 10):
                    update_dict[f'step{i}'] = None
                
                # Update the row with shuffled steps
                for col, value in update_dict.items():
                    shuffled_df.at[idx, col] = value
        
        print("Saving shuffled data...")
        shuffled_df.to_csv(output_file, index=False)
        
        print(f"Successfully saved shuffled data to {output_file}")
        print(f"Total tasks processed: {len(shuffled_df)}")
        
        # Verify shuffle by comparing a few random rows
        print("\nExample of shuffling (first task):")
        print("\nOriginal steps:")
        print(df.iloc[0][step_columns].dropna().tolist())
        print("\nShuffled steps:")
        print(shuffled_df.iloc[0][step_columns].dropna().tolist())
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == "__main__":
    input_file = "wikihowAll-5-dataset before jumbling.csv"
    output_file = "control_flow_dataset.csv"
    
    # Set random seed for reproducibility (optional)
    np.random.seed(42)
    
    shuffle_steps(input_file, output_file)

Reading CSV file...
Shuffling steps for each task...
Saving shuffled data...
Successfully saved shuffled data to control_flow_dataset.csv
Total tasks processed: 31019

Example of shuffling (first task):

Original steps:
['\nScout out locations.,\nPractice safety.,\nGain momentum.,\nCompress your knees.,\nDo an ollie.,\nInitiate the 180.,\nStick the landing.', "\nFind a barrier that's about mid shin level.,\nPractice ollieing over the barrier.,\nGain momentum.,\nCompress your knees.,\nPerform an ollie.,\nInitiate the 180.,\nLand the 180.", '\nUnderstand the faux 180.,\nPractice stationary.,\nGain momentum.,\nHop off and spin.,\nLand the trick.']

Shuffled steps:
['\nScout out locations.,\nPractice safety.,\nGain momentum.,\nCompress your knees.,\nDo an ollie.,\nInitiate the 180.,\nStick the landing.', "\nFind a barrier that's about mid shin level.,\nPractice ollieing over the barrier.,\nGain momentum.,\nCompress your knees.,\nPerform an ollie.,\nInitiate the 180.,\nLand the 180.", '\nUn